# 📖 Lab 2: Search Events

Our second functional requirement: **Users should be able to search for events.**

When a user opens the site, they expect to search for upcoming events by any combination of:
- **Keyword** (event name, performer name)
- **Location** (city)
- **Event type** (concert, sports, comedy, theater)
- **Date range**

## 🏗️ Architecture — Before (Lab 1)

```
┌────────┐         ┌─────────────┐         ┌───────────────┐         ┌──────────────────┐
│ Client │────────>│ API Gateway │────────> │ Event Service │────────>│   PostgreSQL     │
└────────┘         └─────────────┘         └───────────────┘         └──────────────────┘
                                            view(eventId)
```

## 🏗️ Architecture — After (adding Search)

```
┌────────┐         ┌─────────────┐         ┌────────────────┐
│        │────────>│             │────────> │ Search Service │──────┐
│        │         │ API Gateway │         └────────────────┘      │ SQL query
│ Client │         │             │         ┌────────────────┐      │ (ILIKE)
│        │────────>│             │────────> │ Event Service  │──────┤
└────────┘         └─────────────┘         └────────────────┘      │
                                            view(eventId)          v
                                                            ┌──────────────────┐
                    GET /search?term=...&location=...        │   PostgreSQL     │
                                                            │   events          │
                                                            │   venues          │
                                                            │   performers      │
                                                            └──────────────────┘
```

We start with the simplest possible approach: a Search Service that builds a SQL query with filters against the events table. This works for small datasets but has real problems at scale — we'll see why and discuss better options at the end.

## Learning Objectives

- Build a parameterized search query with dynamic filters
- Understand `ILIKE` vs full-text search vs dedicated search engines
- See pagination in action (offset-based)
- Identify why direct SQL search breaks down at scale

## 🛠️ Setup

Make sure PostgreSQL is running from Lab 1:

```bash
cd system-designs/ticketmaster
docker-compose up -d
```

Select the **"Ticketmaster (Python)"** kernel (top-right of the notebook).

In [1]:
import psycopg2
import psycopg2.extras
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "user": "demo",
    "password": "demo",
    "database": "ticketmaster",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM events")
print(f"✅ Connected! {cur.fetchone()[0]} events in the database.")
cur.close()
conn.close()

✅ Connected! 5 events in the database.


## 🔧 Building the Search Service

The Search Service needs to handle a `GET` request with **optional** filter parameters:

```
GET /search?term={term}&location={location}&type={type}&date={date}&page={page}&pageSize={pageSize}
```

Every parameter is optional — if none are provided, we return all events. The tricky part is building the SQL query dynamically based on which filters the user provided.

### Approach 1: Naive ILIKE search (the starting point)

The simplest thing we can do is use `ILIKE` (case-insensitive pattern matching) to filter by keyword, and exact matches for the other fields. This queries the events table directly, joining venue and performer for location and name matching.

In [2]:
def search_events(
    term: str = None,
    location: str = None,
    event_type: str = None,
    date_from: str = None,
    date_to: str = None,
    page: int = 1,
    page_size: int = 10,
) -> dict:
    """
    Simulates the Search Service handler for:
    GET /search?term=...&location=...&type=...&date=...&page=...&pageSize=...
    
    Builds a dynamic SQL query based on which filters are provided.
    Returns Partial<Event>[] — just enough info for search result cards.
    """
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Start with base query — join event + venue + performer
    # We return partial event info (no tickets, no seat map)
    query = """
        SELECT 
            e.id,
            e.name,
            e.event_type,
            e.event_date,
            e.status,
            v.name AS venue_name,
            v.city,
            v.state,
            v.country,
            p.name AS performer_name,
            p.genre
        FROM events e
        JOIN venues v ON e.venue_id = v.id
        JOIN performers p ON e.performer_id = p.id
        WHERE 1=1
    """
    params = []

    # Dynamically add filters based on what the user provided
    if term:
        # ILIKE = case-insensitive LIKE (PostgreSQL-specific)
        # Search across event name AND performer name
        query += " AND (e.name ILIKE %s OR p.name ILIKE %s)"
        params.extend([f"%{term}%", f"%{term}%"])

    if location:
        query += " AND v.city ILIKE %s"
        params.append(f"%{location}%")

    if event_type:
        query += " AND e.event_type = %s"
        params.append(event_type)

    if date_from:
        query += " AND e.event_date >= %s"
        params.append(date_from)

    if date_to:
        query += " AND e.event_date <= %s"
        params.append(date_to)

    # Order by date (upcoming first)
    query += " ORDER BY e.event_date ASC"

    # Pagination: LIMIT + OFFSET
    offset = (page - 1) * page_size
    query += " LIMIT %s OFFSET %s"
    params.extend([page_size, offset])

    start = time.time()
    cur.execute(query, params)
    results = cur.fetchall()
    duration_ms = (time.time() - start) * 1000

    cur.close()
    conn.close()

    return {
        "results": [dict(r) for r in results],
        "page": page,
        "pageSize": page_size,
        "count": len(results),
        "queryTimeMs": round(duration_ms, 2),
    }

print("✅ search_events() function defined.")

✅ search_events() function defined.


## 🧪 Let's Try It!

Let's test different search scenarios a user might perform.

In [3]:
def print_results(label: str, response: dict):
    """Pretty-print search results."""
    print(f"\n{'=' * 70}")
    print(f"🔍 {label}")
    print(f"{'=' * 70}")
    print(f"   Found {response['count']} results (page {response['page']}) in {response['queryTimeMs']}ms\n")

    if not response["results"]:
        print("   No events found.")
        return

    for r in response["results"]:
        date_str = str(r["event_date"])[:10]
        location = f"{r['city']}, {r['state']}" if r["state"] else r["city"]
        print(f"   🎫 {r['name']}")
        print(f"      {r['performer_name']} • {r['venue_name']}, {location}")
        print(f"      📅 {date_str} • 🏷️ {r['event_type']} • 🎵 {r['genre']}")
        print()

# Search 1: No filters — show all events
result = search_events()
print_results("All Events (no filters)", result)


🔍 All Events (no filters)
   Found 5 results (page 1) in 2.11ms

   🎫 The Eras Tour - NYC
      Taylor Swift • Madison Square Garden, New York, NY
      📅 2026-06-15 • 🏷️ concert • 🎵 Pop

   🎫 Championship Game
      Los Angeles Lakers • SoFi Stadium, Inglewood, CA
      📅 2026-07-01 • 🏷️ sports • 🎵 Sports

   🎫 Renaissance World Tour
      Beyoncé • The O2 Arena, London
      📅 2026-08-10 • 🏷️ concert • 🎵 R&B/Pop

   🎫 Kendrick Lamar: Big Steppers
      Kendrick Lamar • Madison Square Garden, New York, NY
      📅 2026-09-20 • 🏷️ concert • 🎵 Hip-Hop

   🎫 Music of the Spheres
      Coldplay • SoFi Stadium, Inglewood, CA
      📅 2026-10-05 • 🏷️ concert • 🎵 Rock



In [4]:
# Search 2: By keyword — matches event name OR performer name
result = search_events(term="taylor")
print_results('Search: term="taylor"', result)

result = search_events(term="coldplay")
print_results('Search: term="coldplay"', result)


🔍 Search: term="taylor"
   Found 1 results (page 1) in 2.99ms

   🎫 The Eras Tour - NYC
      Taylor Swift • Madison Square Garden, New York, NY
      📅 2026-06-15 • 🏷️ concert • 🎵 Pop


🔍 Search: term="coldplay"
   Found 1 results (page 1) in 1.96ms

   🎫 Music of the Spheres
      Coldplay • SoFi Stadium, Inglewood, CA
      📅 2026-10-05 • 🏷️ concert • 🎵 Rock



In [5]:
# Search 3: By location
result = search_events(location="New York")
print_results('Search: location="New York"', result)

# Search 4: By event type
result = search_events(event_type="sports")
print_results('Search: event_type="sports"', result)


🔍 Search: location="New York"
   Found 2 results (page 1) in 1.94ms

   🎫 The Eras Tour - NYC
      Taylor Swift • Madison Square Garden, New York, NY
      📅 2026-06-15 • 🏷️ concert • 🎵 Pop

   🎫 Kendrick Lamar: Big Steppers
      Kendrick Lamar • Madison Square Garden, New York, NY
      📅 2026-09-20 • 🏷️ concert • 🎵 Hip-Hop


🔍 Search: event_type="sports"
   Found 1 results (page 1) in 2.64ms

   🎫 Championship Game
      Los Angeles Lakers • SoFi Stadium, Inglewood, CA
      📅 2026-07-01 • 🏷️ sports • 🎵 Sports



In [6]:
# Search 5: Combined filters — concerts in New York
result = search_events(term="tour", location="New York", event_type="concert")
print_results('Search: term="tour" + location="New York" + type="concert"', result)

# Search 6: By date range
result = search_events(date_from="2026-07-01", date_to="2026-12-31")
print_results('Search: date range Jul-Dec 2026', result)


🔍 Search: term="tour" + location="New York" + type="concert"
   Found 1 results (page 1) in 4.15ms

   🎫 The Eras Tour - NYC
      Taylor Swift • Madison Square Garden, New York, NY
      📅 2026-06-15 • 🏷️ concert • 🎵 Pop


🔍 Search: date range Jul-Dec 2026
   Found 4 results (page 1) in 1.89ms

   🎫 Championship Game
      Los Angeles Lakers • SoFi Stadium, Inglewood, CA
      📅 2026-07-01 • 🏷️ sports • 🎵 Sports

   🎫 Renaissance World Tour
      Beyoncé • The O2 Arena, London
      📅 2026-08-10 • 🏷️ concert • 🎵 R&B/Pop

   🎫 Kendrick Lamar: Big Steppers
      Kendrick Lamar • Madison Square Garden, New York, NY
      📅 2026-09-20 • 🏷️ concert • 🎵 Hip-Hop

   🎫 Music of the Spheres
      Coldplay • SoFi Stadium, Inglewood, CA
      📅 2026-10-05 • 🏷️ concert • 🎵 Rock



## 🔍 Under the Hood: Why ILIKE Is a Problem

Our search works! But let's look at **how** PostgreSQL executes it. The `EXPLAIN ANALYZE` output will reveal a critical issue.

In [ ]:
# Let's see what PostgreSQL is actually doing behind the scenes
conn = get_connection()
cur = conn.cursor()

print("📊 EXPLAIN ANALYZE — keyword search with ILIKE\n")
cur.execute("""
    EXPLAIN ANALYZE
    SELECT e.id, e.name, p.name AS performer_name
    FROM events e
    JOIN venues v ON e.venue_id = v.id
    JOIN performers p ON e.performer_id = p.id
    WHERE e.name ILIKE %s OR p.name ILIKE %s
    ORDER BY e.event_date ASC
""", ("%taylor%", "%taylor%"))

for row in cur.fetchall():
    print(f"  {row[0]}")

cur.close()
conn.close()

print()
print("⚠️  Notice: PostgreSQL uses 'Seq Scan' (sequential scan).")
print("   It reads EVERY row in the table and checks if it matches.")
print("   With 5 events this is instant. With 5 million? Disaster.")

## 📈 Simulating Scale: What Happens with More Data?

Our 5 events search fine. But Ticketmaster has **hundreds of thousands** of events. Let's simulate what happens when the dataset grows by inserting a bunch of events and measuring query time.

In [ ]:
# Insert 50,000 fake events to simulate scale
conn = get_connection()
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM events")
count = cur.fetchone()[0]

if count < 1000:
    print("⏳ Inserting 50,000 fake events for scale testing...")
    cur.execute("""
        INSERT INTO events (name, description, event_type, venue_id, performer_id, event_date)
        SELECT 
            'Event ' || i || ' - ' || (ARRAY['Rock Night', 'Jazz Fest', 'Comedy Hour', 'Sports Finals', 'Pop Concert', 'Tour Stop'])[floor(random() * 6 + 1)::int],
            'Description for event ' || i,
            (ARRAY['concert', 'sports', 'comedy', 'theater'])[floor(random() * 4 + 1)::int],
            (floor(random() * 3) + 1)::int,
            (floor(random() * 8) + 1)::int,
            NOW() + (random() * interval '365 days')
        FROM generate_series(1, 50000) AS i
    """)
    conn.commit()
    print("✅ Done!")
else:
    print(f"✅ Already have {count} events — skipping insert.")

cur.execute("SELECT COUNT(*) FROM events")
print(f"📊 Total events in database: {cur.fetchone()[0]}")

cur.close()
conn.close()

In [ ]:
# Now let's measure search time with 50k events
print("⏱️  Search performance with 50,000+ events:\n")

searches = [
    ("No filters",           {}),
    ("term='Taylor'",        {"term": "Taylor"}),
    ("term='Rock Night'",    {"term": "Rock Night"}),
    ("location='New York'",  {"location": "New York"}),
    ("type='concert'",       {"event_type": "concert"}),
    ("Combined filters",     {"term": "Jazz", "location": "London", "event_type": "concert"}),
]

print(f"{'Search':<25} {'Results':<10} {'Time (ms)'}")
print("-" * 50)

for label, params in searches:
    result = search_events(**params)
    print(f"{label:<25} {result['count']:<10} {result['queryTimeMs']}")

print()
print("💡 With small data, ILIKE is fast enough.")
print("   But ILIKE does a FULL TABLE SCAN — it scales linearly with data size.")
print("   At 1M+ events, these queries will blow past our 500ms SLA.")

## 🧹 Cleanup: Remove Fake Events

Let's remove the scale test data so it doesn't interfere with other labs.

In [ ]:
# Remove the fake events (keep only our original 5)
conn = get_connection()
cur = conn.cursor()
cur.execute("DELETE FROM events WHERE id > 5")
conn.commit()
cur.execute("SELECT COUNT(*) FROM events")
print(f"✅ Cleaned up. {cur.fetchone()[0]} events remaining.")
cur.close()
conn.close()

## 🤔 What's Wrong with This Approach?

Our naive search works, but it has serious problems at scale:

| Problem | Why It Matters |
|---------|---------------|
| **`ILIKE` = full table scan** | PostgreSQL must check every row — `O(n)` with data size. Can't use indexes for `%term%` patterns. |
| **No relevance ranking** | Results are ordered by date, not by how well they match the search. A user searching "Taylor" gets results in chronological order, not by relevance. |
| **No typo tolerance** | Searching "Tayor Swift" returns nothing. Real search engines handle fuzzy matching. |
| **Joins on every query** | Every search requires joining events + venues + performers. At high QPS this hammers the database. |
| **Offset pagination is slow** | `OFFSET 10000` means PostgreSQL scans and discards 10,000 rows. Gets slower as you paginate deeper. |

### What would a production system use?

In the non-functional requirements deep dives, we'd evolve this to use **Elasticsearch** — a dedicated search engine that:
- Uses **inverted indexes** for O(1) text lookups (not O(n) table scans)
- Supports **fuzzy matching** and **typo tolerance**
- Ranks results by **relevance score**
- Handles **millions of documents** with sub-100ms latency
- Can be independently scaled from the primary database

The Event Service would write events to both PostgreSQL (source of truth) and Elasticsearch (optimized for search). The Search Service would query Elasticsearch instead of PostgreSQL.

```
Write path:  Event Service ──> PostgreSQL ──(sync)──> Elasticsearch
Read path:   Search Service ──> Elasticsearch (not PostgreSQL!)
```

But that's a deep dive for later. For now, our SQL-based search satisfies the functional requirement — and we understand exactly where it will break.

## ✅ Summary

- Built a `search_events()` function that dynamically builds SQL queries from optional filters
- Tested keyword, location, type, date range, and combined searches
- Identified that `ILIKE` causes full table scans — O(n) performance
- Understood why production systems use Elasticsearch for search
- Next up: **Lab 3 — Booking Tickets** (where consistency really matters)